In [ ]:
# this notebook will be used for training a model on brain tumor Segmentation
# The dataset: Moani
# deep learning model : Unet

In [1]:
pip install modal

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!python -m modal

Usage: python -m modal [OPTIONS] COMMAND [ARGS]...

  Modal is the fastest way to run code in the cloud.

  See the website at https://modal.com/ for documentation and more information
  about running code on Modal.

Commands:
  run     Run a Modal function or local entrypoint.
  serve   Expose Web Functions with hot-reloading on code changes.
  deploy  Deploy a Modal application.
  shell   Run a command or interactive shell inside a Modal container.
  curl    Send an authenticated request to a Modal endpoint.

Compute:
  app        Manage deployed and running Apps.
  container  Manage and connect to running containers.
  endpoint   Create and manage LLM inference endpoints.
  function   Inspect Modal Functions.
  server     Inspect Modal Servers.

Storage:
  image   Manage Images.
  dict    Manage `modal.Dict` objects and inspect their contents.
  secret  Manage secrets.
  queue   Manage `modal.Queue` objects and inspect their contents.
  volume  Read and edit `modal.Volume` volumes.


In [2]:
#install MONAI package
!pip install -q monai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.5 MB/s eta 0:00:0000:0100:01


In [6]:
#import libraries
import torch
from monai.transforms import (
Compose,
LoadImaged,
EnsureChannelFirstd,
NormalizeIntensityd,
RandSpatialCropd,
ToTensord,
AsDiscrete,
MapLabelValued,
RandCropByPosNegLabeld
)
from torch.utils.data import DataLoader
from monai.apps import DecathlonDataset
import os
from monai.losses import DiceCELoss
from monai.networks.nets import UNet
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

from monai.metrics import DiceMetric, HausdorffDistanceMetric
from monai.data import decollate_batch, DataLoader
from monai.inferers import sliding_window_inference

In [7]:
transform = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    ToTensord(keys=["image", "label"]),
    MapLabelValued(
        keys=["label"],
        orig_labels=[0, 1, 2, 4],
        target_labels=[0, 1, 2, 3],
    ),
])

nvImageCodec is not available; skipping pydicom decoder plugin registration.
/usr/local/lib/python3.13/dist-packages/monai/data/image_reader.py:1166: UserWarning: NvImgCodecPydicomReader: nvImageCodec decoder plugin did not register successfully. Falling back to default pydicom decoders.
  warnings.warn(


In [12]:
task = "Task01_BrainTumour"
root_dir = "./kaggle/working/monai_dataset"
os.makedirs(root_dir, exist_ok=True)

# 1. Compose the base transform + the random crop for TRAINING
train_transforms = Compose([
    transform,                             
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1,
        neg=1,
        num_samples=2,
        image_key="image",
        image_threshold=0,
    ),
])

# 2. Training dataset — use train_transforms
train_ds = DecathlonDataset(
    root_dir=root_dir,
    task=task,
    section="training",
    download=True,
    cache_rate=0,
    transform=train_transforms,
)

# 3. no random crop for validation dataset
val_ds = DecathlonDataset(
    root_dir=root_dir,
    task=task,
    section="validation",
    download=False,
    transform=transform,
)

Task01_BrainTumour.tar: 7.09GB [03:57, 32.0MB/s]                               


2026-10-16 07:10:33,684 - INFO - Verified 'Task01_BrainTumour.tar', md5: 240a19d752f0d9e9101544901065d872.
2026-10-16 07:10:42,049 - INFO - Downloaded: kaggle/working/monai_dataset/Task01_BrainTumour.tar
2026-10-16 07:10:42,050 - INFO - Writing into directory: kaggle/working/monai_dataset.


Loading dataset: 100%|██████████| 96/96 [02:18<00:00,  1.44s/it]


In [13]:
len(train_ds)

388

In [14]:
len(val_ds)

96

In [15]:
#Training Phase
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [16]:

# 2. Create both data loaders
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

print(f"Training batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

Training batches: 194 | Validation batches: 96


In [ ]:
#define My model:
model = UNet(
    spatial_dims = 3,
    in_channels = 4,
    out_channels = 4,
    channels = (16, 32, 64, 128, 256),
    strides = (2, 2, 2, 2),
    num_res_units = 2,
).to(device)

#define Loss Function and Optimizer
#I was using sigmoid activation which treats every single class independenly like multilabel classification task
#I changed it to softmax
loss_fct = DiceCELoss(to_onehot_y = True, softmax = True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4)

scaler = torch.cuda.amp.GradScaler()

num_epoch = 30
for epoch in range(num_epoch):
    model.train()
    epoch_loss = 0
    step = 0

    for batch_data in train_loader:
        step += 1
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            outputs = model(inputs)
            loss = loss_fct(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
    print(f"Epoch {epoch + 1}/{num_epoch}, Loss : {epoch_loss/step: .4f}")

print("Training phase is done")

#Save the Trained weights
torch.save(model.state_dict(), "segmentation_model.pth")
print("Model weights are successfully saved")
    


/tmp/ipykernel_49/3927824757.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_49/3927824757.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1/30, Loss :  1.8289
Epoch 2/30, Loss :  1.2404
Epoch 3/30, Loss :  0.8912
Epoch 4/30, Loss :  0.7414
Epoch 5/30, Loss :  0.6635
Epoch 6/30, Loss :  0.6174
Epoch 7/30, Loss :  0.5670
Epoch 8/30, Loss :  0.5570
Epoch 9/30, Loss :  0.5195
Epoch 10/30, Loss :  0.4995
Epoch 11/30, Loss :  0.4876
Epoch 12/30, Loss :  0.4803
Epoch 15/30, Loss :  0.4394
Epoch 16/30, Loss :  0.4446
Epoch 17/30, Loss :  0.4350
Epoch 18/30, Loss :  0.4233
Epoch 19/30, Loss :  0.4231
Epoch 20/30, Loss :  0.4186
Epoch 21/30, Loss :  0.4103
Epoch 22/30, Loss :  0.4078
Epoch 23/30, Loss :  0.4140
Epoch 24/30, Loss :  0.3980
Epoch 25/30, Loss :  0.4059


In [19]:
model.load_state_dict(torch.load("/kaggle/working/segmentation_model.pth"))
model.to(device)
model.eval()

dice_metric = DiceMetric(include_background=False, reduction="mean")
hausdorff_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")

post_trans = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

with torch.no_grad():
    for val_ds in val_loader:  # Assumes you have a validation data loader configured
        val_inputs, val_labels = val_ds["image"].to(device), val_ds["label"].to(device)
        
        # Forward pass
        val_outputs = sliding_wnidow_inference(val_inputs, (96, 96, 96), 4, model)
        
        # Decollate batch tensors into lists for MONAI metrics
        val_outputs_list = [post_trans(i) for i in decollate_batch(val_outputs)]
        val_labels_list = [post_label(i) for i in decollate_batch(val_labels)]
        
        # Update metric accumulators
        dice_metric(y_pred=val_outputs_list, y=val_labels_list)
        hausdorff_metric(y_pred=val_outputs_list, y=val_labels_list)

    # Compute final aggregated scores across the entire validation set
    mean_dice = dice_metric.aggregate().item()
    mean_hausdorff = hausdorff_metric.aggregate().item()
    
    # Reset metrics for clean state
    dice_metric.reset()
    hausdorff_metric.reset()

print(f"Mean Dice Score: {mean_dice:.4f}")
print(f"Mean Hausdorff Distance (HD95): {mean_hausdorff:.4f} voxels")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/segmentation_model.pth'